# V9 Phase 2: Extract Visual Features with ResNet-50

## Goal
Extract 768-dimensional visual features from hieroglyph images using ResNet-50 (pre-trained on ImageNet).

## Process
1. Load ResNet-50 model (remove classification head → 2048d features)
2. Add projection layer: 2048d → 768d (to match FastText dimensions)
3. Extract features for each Gardiner code
4. Average features across multiple images of the same Gardiner code
5. Save: `visual_embeddings.pkl` → {Gardiner_code: 768d_vector}

## Why ResNet-50?
- Pre-trained on ImageNet (general visual features)
- Proven effective for hieroglyph recognition
- 2048d features are rich enough to capture glyph shapes

In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from collections import defaultdict
import numpy as np
import pickle
from tqdm.auto import tqdm

# Setup paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_DIR = PROJECT_ROOT / 'data/raw/hieroglyph_dataset/Dataset'
OUTPUT_DIR = PROJECT_ROOT / 'data/processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Dataset Directory: {DATASET_DIR}')
print(f'Output Directory: {OUTPUT_DIR}')

Project Root: /Users/crashy/Development/heiroglyphy/heiro_v9_use_visuals_again
Dataset Directory: /Users/crashy/Development/heiroglyphy/heiro_v9_use_visuals_again/data/raw/hieroglyph_dataset/Dataset
Output Directory: /Users/crashy/Development/heiroglyphy/heiro_v9_use_visuals_again/data/processed


## Load ResNet-50 Model

In [3]:
# Load pre-trained ResNet-50
print('Loading ResNet-50 (pre-trained on ImageNet)...')
resnet = models.resnet50(pretrained=True)

# Remove the classification head (fc layer)
# This gives us 2048d features from the avgpool layer
resnet = nn.Sequential(*list(resnet.children())[:-1])

# Add projection layer: 2048d → 768d
projection = nn.Linear(2048, 768)

# Combine into feature extractor
feature_extractor = nn.Sequential(
    resnet,
    nn.Flatten(),
    projection
)

# Set to evaluation mode
feature_extractor.eval()

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
feature_extractor = feature_extractor.to(device)

print(f'✓ Model loaded on device: {device}')

Loading ResNet-50 (pre-trained on ImageNet)...
✓ Model loaded on device: mps


## Define Image Preprocessing

In [4]:
# ImageNet normalization (ResNet-50 was trained with these)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print('✓ Image preprocessing pipeline ready')

✓ Image preprocessing pipeline ready


## Extract Features from All Images

In [5]:
# Collect all image paths
train_images = list((DATASET_DIR / 'train').rglob('*.png'))
test_images = list((DATASET_DIR / 'test').rglob('*.png'))
all_images = train_images + test_images

print(f'Found {len(train_images)} training images')
print(f'Found {len(test_images)} test images')
print(f'Total: {len(all_images)} images')

Found 1743 training images
Found 455 test images
Total: 2198 images


In [6]:
# Extract features for each image, grouped by Gardiner code
gardiner_features = defaultdict(list)

print('Extracting features...')
with torch.no_grad():
    for img_path in tqdm(all_images, desc='Processing images'):
        # Extract Gardiner code from filename
        # Format: 070029_D21.png → D21
        gardiner_code = img_path.stem.split('_')[1]
        
        # Skip UNKNOWN glyphs
        if gardiner_code == 'UNKNOWN':
            continue
        
        try:
            # Load and preprocess image
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)
            
            # Extract features
            features = feature_extractor(img_tensor)
            
            # Store features (move to CPU and convert to numpy)
            gardiner_features[gardiner_code].append(features.cpu().numpy()[0])
            
        except Exception as e:
            print(f'Error processing {img_path}: {e}')
            continue

print(f'\n✓ Extracted features for {len(gardiner_features)} unique Gardiner codes')

Extracting features...


Processing images:   0%|          | 0/2198 [00:00<?, ?it/s]


✓ Extracted features for 115 unique Gardiner codes


## Average Features per Gardiner Code

In [7]:
# Average features across multiple images of the same Gardiner code
visual_embeddings = {}

for gardiner_code, features_list in gardiner_features.items():
    # Average all feature vectors for this Gardiner code
    avg_features = np.mean(features_list, axis=0)
    visual_embeddings[gardiner_code] = avg_features

print(f'Created {len(visual_embeddings)} visual embeddings')
print(f'\nTop 10 Gardiner codes by image count:')
sorted_codes = sorted(gardiner_features.items(), key=lambda x: len(x[1]), reverse=True)[:10]
for code, features in sorted_codes:
    print(f'  {code}: {len(features)} images → averaged to 768d vector')

Created 115 visual embeddings

Top 10 Gardiner codes by image count:
  M17: 364 images → averaged to 768d vector
  G43: 197 images → averaged to 768d vector
  G17: 195 images → averaged to 768d vector
  D21: 183 images → averaged to 768d vector
  I9: 146 images → averaged to 768d vector
  E34: 122 images → averaged to 768d vector
  D36: 59 images → averaged to 768d vector
  D35: 57 images → averaged to 768d vector
  D46: 50 images → averaged to 768d vector
  N35: 45 images → averaged to 768d vector


## Save Visual Embeddings

In [8]:
# Save to pickle file
output_path = OUTPUT_DIR / 'visual_embeddings_768d.pkl'

with open(output_path, 'wb') as f:
    pickle.dump(visual_embeddings, f)

print(f'✓ Saved visual embeddings to: {output_path}')
print(f'  File size: {output_path.stat().st_size / 1024:.1f} KB')
print(f'  Embeddings: {len(visual_embeddings)} Gardiner codes')
print(f'  Dimension: 768d per code')

✓ Saved visual embeddings to: /Users/crashy/Development/heiroglyphy/heiro_v9_use_visuals_again/data/processed/visual_embeddings_768d.pkl
  File size: 349.6 KB
  Embeddings: 115 Gardiner codes
  Dimension: 768d per code


## Verify Embeddings

In [9]:
# Load and verify
with open(output_path, 'rb') as f:
    loaded_embeddings = pickle.load(f)

# Check a few examples
print('Sample embeddings:')
for i, (code, vec) in enumerate(list(loaded_embeddings.items())[:5]):
    print(f'  {code}: shape={vec.shape}, mean={vec.mean():.4f}, std={vec.std():.4f}')

print(f'\n✓ Visual embeddings ready for fusion!')

Sample embeddings:
  F18: shape=(768,), mean=0.0056, std=0.2949
  F29: shape=(768,), mean=0.0049, std=0.2975
  F16: shape=(768,), mean=0.0046, std=0.3057
  D46: shape=(768,), mean=-0.0011, std=0.2943
  F26: shape=(768,), mean=0.0122, std=0.3496

✓ Visual embeddings ready for fusion!


## Summary

**Completed:**
- Extracted ResNet-50 features from all hieroglyph images
- Averaged features per Gardiner code
- Saved 768d visual embeddings dictionary

**Next Steps:**
1. Create Transliteration → Gardiner Code mapping (using V6 lexicon)
2. Fuse FastText (text) + Visual embeddings
3. Train alignment and evaluate